In [ ]:
!pip install numpy==2.2.6 \
             torch==2.6.0 \
             torchvision==0.21.0 \
             nltk==3.9.4 \
             spacy==3.8.14 \
             matplotlib==3.10.9 \
             tqdm==4.67.3 \
             transformers==5.9.0 \
             datasets==4.8.5 \
             scikit-learn==1.7.2 
!python -m spacy download en_core_web_sm

## Халюцинации
Ти си изследовател в лаборатория за изкуствен интелект и ръководиш важен проект за създаване на мащабен мултимодален набор от данни. За да автоматизирате и ускорите процеса, екипът ти е използвал Голям езиков модел (LLM), който да генерира текстови описания (captions) за стотици хиляди изображения.

За съжаление, при анотиране на данни от LLM често се получават сериозни отклонения. Вследствие на дефекти в декодирането или липса на достатъчно визуален контекст, моделът понякога започва да "халюцинира" обекти, които изобщо не съществуват на снимката (например, твърди, че вижда куче, когато на нея има котка), или генерира напълно несвързан и нелогичен текст (incoherent text). Твоята задача е да изчистиш този набор от данни, като създадеш стабилен филтър, който да разпознава дали дадено описание е коректно, халюцинирано или напълно несвързано.

### Задача и ограничения
Системата, която разработваш, ще бъде интегрирана в среда с изключително строги лимити за памет и изчислително време. Затова главният инженер на проекта е наложил следните железни правила за твоето решение:
- Фиксиран класификатор: Задължително трябва да използваш предварително зададения **GradientBoostingClassifier**. Абсолютно е забранено да променяш неговите хиперпараметри или да добавяш други модели (като невронни мрежи) към пайплайна за класификация.

- Брой характеристики: Имаш право да конструираш и подадеш максимум **16 features** (характеристики) към класификатора. Ще трябва внимателно да подбереш кои метрики носят най-много семантична и визуална информация.

- Без обучение (Unsupervised): За конструиране на характеристиките е разрешено използването само на методи `без обучение (unsupervised)`. Етикетите от данните могат да се използват единствено за анализ и селекция, но не и за трениране на вашия пайплайн за извличане на характеристики.

- Библиотеки: За извличането на тези характеристики можеш да използваш единствено:
    - `transfomers` – за извличане на ембединги и пресмятане на семантичната близост между изображението и текста, но само и единствено зареденият модел.
    - `nltk` – Разрешени са единствено и само следните подмодули:
        - `nltk.tokenize`
        - `nltk.util`
        - `nltk.lm`
    - `torch`и `numpy` - За обработка на ембедингите, но не и за трениране
    - `spacy` – забранено е директното импортиране и използване на spacy в твоя код. Цялата лингвистична информация (Parts-Of-Speech тагове, токени) трябва да бъде извлечена единствено чрез предоставената помощна функция от старшите изследователи: `get_linguistic_features(caption)`.

Никакви други външни библиотеки не са позволени!

### Данни
Наборът от данни се зарежда чрез библиотеката `datasets` и съдържа следните ключови полета за всеки пример:
- `image`: Изображението (PIL Image).
- `caption`: Текстовото описание, генерирано от LLM.
- `label`: Целевият клас, който трябва да предскажеш (0, 1 или 2).

Етикетите отговарят на трите възможни състояния на текста:
- Клас 0: Коректно описание.
- Клас 1: Халюцинация (подменено съществително).
- Клас 2: Несвързан текст (разбъркани думи).

За твоите експерименти разполагаш с тренировъчен (train - 2400 примера) и валидационен (val - 800 примера без етикет) сплит, които са балансирани. Забранено е използването на външни данни. Препоръчително е да разгледаш суровите данни, за да придобиеш интуиция за разликите между класовете, преди да започнеш с извличането на характеристиките.

### Налични инструменти
За да те улеснят поне малко в задачата, старшите изследователи от екипа са ти подготвили помощна функция - `get_linguistic_features(caption)`. Тя използва spacy и автоматично връща частите на речта (Part-of-Speech tags) на думите в подаденото изречение.
#### Описание на елементите, връщани от `get_linguistic_features()`

| Ключ | Тип | Описание | Възможни стойности и значение |
|--------|--------|--------|--------|
| `tokens` | `list[str]` | Списък от всички токени (думи, числа, препинателни знаци и др.) в текста. | Произволни текстови низове, например: `"cat"`, `"running"`, `"."`, `"2025"`. Всеки елемент представлява оригинален токен от текста. |
| `pos` | `list[str]` | Универсална граматична категория (Part-of-Speech) за всеки токен. | `ADJ` – прилагателно име;<br>`ADP` – предлог;<br>`ADV` – наречие;<br>`AUX` – спомагателен глагол;<br>`CCONJ` – съчинителен съюз;<br>`DET` – определител;<br>`INTJ` – междуметие;<br>`NOUN` – съществително име;<br>`NUM` – числително;<br>`PART` – частица;<br>`PRON` – местоимение;<br>`PROPN` – собствено име;<br>`PUNCT` – препинателен знак;<br>`SCONJ` – подчинителен съюз;<br>`SYM` – символ;<br>`VERB` – глагол;<br>`X` – друг/неразпознат тип;<br>`SPACE` – интервал. |
| `tag` | `list[str]` | Подробен граматичен етикет според Penn Treebank. | `NN` – съществително, ед.ч.;<br>`NNS` – съществително, мн.ч.;<br>`NNP` – собствено име, ед.ч.;<br>`NNPS` – собствено име, мн.ч.;<br>`VB` – глагол, основна форма;<br>`VBD` – глагол, минало време;<br>`VBG` – глагол в форма *-ing*;<br>`VBN` – минало причастие;<br>`VBP` – сегашно време (без 3 л. ед.ч.);<br>`VBZ` – сегашно време (3 л. ед.ч.);<br>`JJ` – прилагателно;<br>`JJR` – сравнителна степен;<br>`JJS` – превъзходна степен;<br>`RB` – наречие;<br>`RBR` – сравнителна степен на наречие;<br>`RBS` – превъзходна степен на наречие;<br>`PRP` – лично местоимение;<br>`PRP$` – притежателно местоимение;<br>`DT` – определител;<br>`IN` – предлог или подчинителен съюз;<br>`CC` – съчинителен съюз;<br>`CD` – число;<br>`.` – край на изречение;<br>`,` – запетая. |
| `is_stop` | `list[bool]` | Показва дали токенът е стоп-дума. | `True` – токенът е стоп-дума (напр. *the*, *and*, *is*, *of*);<br>`False` – токенът не е стоп-дума. |
| `lemma` | `list[str]` | Основна (речникова) форма на думата. | Текстов низ, например:<br>`"cats" → "cat"`;<br>`"running" → "run"`;<br>`"are" → "be"`;<br>`"better" → "good"`. |

##### Пример

Вход:

```python
caption = "The cats are running quickly."
```

Изход:

```python
{
    "tokens": ["The", "cats", "are", "running", "quickly", "."],
    "pos": ["DET", "NOUN", "AUX", "VERB", "ADV", "PUNCT"],
    "tag": ["DT", "NNS", "VBP", "VBG", "RB", "."],
    "is_stop": [True, False, True, False, False, False],
    "lemma": ["the", "cat", "be", "run", "quickly", "."]
}
```

Твоята цел е креативно да комбинираш тази лингвистична информация (структура на изречението, брой съществителни, граматическа коректност) с визуалното разбиране на CLIP (дали текстът реално отговаря на снимката), за да избереш перфектните 16 характеристики и да спасиш данните на лабораторията!
### Оценяване
Оценяването се изпълнява на скрит набор от данни (test split), а водещата метрика за класиране на моделите е F1 macro, за да се гарантира еднакво добро разпознаване и на трите класа. Можете да видите своя предварителен резултат като submit-нете предсказания за валидационния сплит в Kaggle. Взема се последната тетрадка, която сте запазили с опцията `Save & Run All (Commit)`.

In [ ]:
import numpy as np
import torch
import nltk
import spacy
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
from datasets import load_dataset

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [ ]:
# 1. NLTK Setup
print("Изтегляне на NLTK ресурси (punkt)...")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# 2. spaCy Setup
print("Зареждане на spaCy (en_core_web_sm)...")
nlp = spacy.load("en_core_web_sm")
def get_linguistic_features(caption: str) -> dict:
    """
    Помощна функция, която предоставя само сурови лингвистични данни.
    """
    doc = nlp(caption)
    return {
        "tokens": [t.text for t in doc],
        "pos": [t.pos_ for t in doc],
        "tag": [t.tag_ for t in doc],
        "is_stop": [t.is_stop for t in doc],
        "lemma": [t.lemma_ for t in doc]
    }

# 3. Зареждане на данните
print("Зареждане на тренировъчните и валидационните данни...")
labeled_data = load_dataset("delyanboychev/hallucinations_problem_data")
print(f"Успешно заредени сплитове: {list(labeled_data.keys())}")

# 4. CLIP Setup
print("Инициализация на CLIP модела...")
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Използвано устройство (Device): {device.upper()}")

model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to(device)
model.eval() 

print("Всички ресурси и данни са заредени успешно!")

### Визуализация

In [ ]:
# Помощна функция, предоставена от старшите изследователи :)
def get_pos_tags(text):
    """
    Връща структура, подходяща за визуализация (text) 
    и за математическа обработка (dict).
    """
    features = get_linguistic_features(text)
    
    readable_tags = [f"{token} ({pos})" for token, pos in zip(features["tokens"], features["pos"])]
    return readable_tags

# Речник за по-лесно разчитане на класовете
label_map = {
    0: "0: Оригинал (Коректно)",
    1: "1: Сменен обект (Халюцинация)",
    2: "2: Разбъркани думи (Несвързано)"
}

# Визуализираме точно 3 примера
for i in range(3):
    sample = labeled_data["train"][i]
    image = sample["image"]
    caption = sample["caption"]
    label = sample["label"]
    
    print("\n" + "="*60)
    print(f"Пример {i+1}")
    print(f"Етикет (Label): {label_map.get(label, 'Неизвестен')}")
    print(f"Текст (Caption): {caption}")
    print("-" * 60)
    
    # Извикваме помощната функция
    pos_tags = get_pos_tags(caption)
    print("Части на речта (POS Tags):")
    print(" | ".join(pos_tags))
    print("="*60)
    
    # Показваме самото изображение
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.title(f"Label: {label}")
    plt.axis("off")
    plt.show()

### Решение

In [ ]:
# !!! ИМАТЕ ПРАВО ДА ИМПОРТИРАТЕ СПЕЦИФИЧНИ МОДУЛИ, КАКТО И ДА ЗАПАЗВАТЕ ГЛОБАЛНИ ПРОМЕНЛИВИ, НО САМО В ТАЗИ КЛЕТКА !!!

def extract_features(dataset_split, batch_size=64, is_train=False):
    """
    Извлича характеристики на партиди (batches).
    Връща САМО матрицата X (с характеристики, макс 64 колони).
    is_train може да ви помогне, ако трябва да си запазите някакви състояние извлечени от
    сета за трениране, които може да използвате по време на тестване.
    """
    
    # =========================================================
    # ТУК ТРЯБВА ДА НАПИШЕТЕ ВАШЕТО РЕШЕНИЕ
    # =========================================================
    # Дадени са базови функции, за да ви помогнат с решаването!

    def get_simple_nltk_feature(text: str) -> float:
        """Пример: Лексикално разнообразие чрез NLTK."""
        tokens = nltk.word_tokenize(text.lower())
        return len(set(tokens)) / len(tokens) if tokens else 0.0

    def get_noun_density(ling_data: dict) -> float:
        """Пример: Плътност на съществителните, използвайки ling_data."""
        nouns = [1 for pos in ling_data["pos"] if pos == "NOUN"]
        return sum(nouns) / len(ling_data["tokens"]) if ling_data["tokens"] else 0.0
    
    # =========================================================
    
    X = []

    for i in tqdm(range(0, len(dataset_split), batch_size), desc="Извличане на характеристики"):
        batch = dataset_split[i : i + batch_size]
        images = batch["image"]
        captions = batch["caption"]

        # --- Стъпка 1: Извличане на лингвистични примитиви ---
        # Използваме задължителната помощна функция
        ling_results = [get_linguistic_features(cap) for cap in captions]

        # --- Стъпка 2: Изчисляване на CLIP характеристиките ---
        inputs = processor(
            text=captions, images=images, return_tensors="pt", padding=True
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

            # Нормализиране на ембедингите
            image_embeds = outputs.image_embeds / outputs.image_embeds.norm(p=2, dim=-1, keepdim=True)
            text_embeds = outputs.text_embeds / outputs.text_embeds.norm(p=2, dim=-1, keepdim=True)

            # Batched Cosine Similarity
            cosine_sims = (image_embeds * text_embeds).sum(dim=-1).cpu().numpy()

        # --- Стъпка 3: Сглобяване на финалния вектор ---
        for idx, sim in enumerate(cosine_sims):
            
            # ВНИМАНИЕ: Добавете вашите нови характеристики в този списък!
            # Максимален лимит: 16 елемента.
            
            # Извличаме готови данни за текущия пример
            ling_data = ling_results[idx]
            
            # Изчисляваме характеристиките
            nltk_feat = get_simple_nltk_feature(captions[idx])
            noun_feat = get_noun_density(ling_data)
            
            feature_vector = [sim, nltk_feat, noun_feat]
            
            X.append(feature_vector)

    return np.array(X)

### Оценяване

In [ ]:
# ==========================================
# ВНИМАНИЕ: АБСОЛЮТНО Е ЗАБРАНЕНО ДА ПРОМЕНЯТЕ ТОЗИ БЛОК!
# ==========================================
BATCH_SIZE = 64

print("--- Обработка на Train Split ---")
# Взимаме етикетите за трениране
y_train = np.array(labeled_data["train"]["label"])

safe_train_data = labeled_data["train"].remove_columns("label")
safe_val_data = labeled_data["val"].remove_columns("label")


X_train = extract_features(safe_train_data, batch_size=BATCH_SIZE, is_train=True)


print("\n--- Обработка на Val Split ---")
X_val = extract_features(safe_val_data, batch_size=BATCH_SIZE, is_train=False)


assert X_train.shape[1] <= 16, f"Грешка: Използвате {X_train.shape[1]} характеристики, а лимитът е 16!"
assert X_val.shape[1] <= 16, f"Грешка: Валидационният сет има {X_val.shape[1]} характеристики!"

print("\nОбучение на фиксирания Gradient Boosting Classifier...")

# Използваме пайплайн със StandardScaler за числена стабилност
clf = make_pipeline(
    StandardScaler(),
    GradientBoostingClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        max_features="sqrt",
        min_samples_leaf=5,
        random_state=42,
    ),
)

# Тренираме модела
clf.fit(X_train, y_train)

# Предсказваме върху валидационния сет
print("\nГенериране на предсказания...")
y_val_pred = clf.predict(X_val)

# Създаване на файл за предаване в Kaggle
print("Създаване на submission.csv...")

submission_df = pd.DataFrame({
    "id": labeled_data["val"]["id"], 
    "label": y_val_pred
})

submission_df.to_csv("submission.csv", index=False)
print("\n" + "=" * 50)
print("ГОТОВО! Файлът 'submission.csv' е запазен успешно.")
print("Можете да го свалите и предадете в Kaggle, за да видите резултата си!")
print("=" * 50)